# NL2SQL Inference Pipeline

End-to-end demo: load a user CSV/SQLite file, ask a question in plain English, generate SQL with the fine-tuned T5 model, and run it on the uploaded data.

Core logic lives in `inference.py` (shared with the Streamlit chatbot in `app.py`).

## 1. Setup

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
import pandas as pd

from inference import NL2SQLPipeline, load_user_data, PROJECT_ROOT, CHECKPOINT_DIR

print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Exists: {CHECKPOINT_DIR.exists()}")

## 2. Load Model

In [ ]:
pipeline = NL2SQLPipeline()
print("Model ready.")

## 3. Sanity Check

The checkpoint emits WikiSQL structs; `inference.py` converts them to SQL.

In [ ]:
demo_headers = ["Player", "Position", "Team", "Points", "Assists"]
demo_schema = {
    "tables": [{
        "columns": [{"name": h, "type": "text"} for h in demo_headers],
    }]
}
raw, sql, retrieved = pipeline.generate_sql(
    "What are the points for Alice?",
    demo_schema,
)
print(f"Raw: {raw}")
print(f"SQL: {sql}")
print(f"Retrieved {len(retrieved)} example(s)")
assert sql.strip().upper().startswith("SELECT")

## 4. Load Your Data

Supports `.csv` and `.db` / `.sqlite` files.

In [ ]:
MOCK_CSV = """Player,Position,Team,Points,Assists
Alice,Guard,Lakers,28,7
Bob,Forward,Warriors,19,4
Carol,Center,Heat,14,2
Dave,Guard,Lakers,22,9
Eve,Forward,Warriors,31,5"""

mock_path = PROJECT_ROOT / "demo_players.csv"
mock_path.write_text(MOCK_CSV)
schema = load_user_data(mock_path)

for t in schema["tables"]:
    print(f"Table '{t['table_name']}': {len(t['columns'])} cols, {t['row_count']} rows")

## 5. Ask Questions

In [ ]:
questions = [
    "What are the points for Alice?",
    "How many players are on the Lakers?",
    "What is the maximum points scored?",
    "Show me the team where Player is Eve",
]

for q in questions:
    print(f"\nQuestion: {q}")
    out = pipeline.query(q, schema)
    print(f"SQL: {out['sql']}")
    if isinstance(out["result"], pd.DataFrame):
        display(out["result"])
    else:
        print(out["result"])

## 6. Chatbot GUI

For an interactive UI with file upload and chat history, run from the project root:

```bash
pip install streamlit
streamlit run app.py
```